## J40 Paper Tabular Data Creation

In [1]:
# Load packages 
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns         
import numpy as np
import os

# Remove max columns
pd.set_option('display.max_columns', None)

In [2]:
# Read in new Geoda Data
# base_dir = "/capstone/justice40"
base_dir = "~/MEDS/justice40/data-exploration/data"

# December 2025 data
# burdens = gpd.read_file(os.path.join(base_dir, "geoda-dec-2025", "december-geoda-output.geojson"))
geoda_raw = gpd.read_file(os.path.join(base_dir, "geoda-dec-2025", "december-geoda-output-indicators.geojson"))

In [6]:
# Rename columns and slim
geoda_small = geoda_raw[['GEOID10', 'SN_C', 'SF', 'CF', 'TC', 'CC', 'G_STR', 'C_ID', 'PP_VAL', 'G_STR_indicator', 'C_ID_indicator', 'PP_VAL_indicator', 'geometry']]

geoda_small = geoda_small.rename(columns={
    'G_STR':'g_burd',
    'C_ID':'id_burd',
    'PP_VAL':'p_burd',
    'G_STR_indicator':'g_ind',
    'C_ID_indicator':'id_ind',
    'PP_VAL_indicator':'p_ind'
})
geoda_small.head()

,GEOID10,SN_C,SF,CF,TC,CC,g_burd,id_burd,p_burd,g_ind,id_ind,p_ind,geometry
0,01073001100,1,Alabama,Jefferson County,5,4.0,0.000050,1,0.0002,0.000052,1,0.0006,"MULTIPOLYGON (((-86.88244 33.55233, -86.88187 ..."
1,01073001400,1,Alabama,Jefferson County,9,6.0,0.000070,1,0.0001,0.000086,1,0.0001,"MULTIPOLYGON (((-86.84088 33.52759, -86.83782 ..."
2,01073002000,1,Alabama,Jefferson County,8,3.0,0.000051,1,0.0003,0.000065,1,0.0001,"MULTIPOLYGON (((-86.71390 33.53930, -86.71435 ..."
3,01073003802,1,Alabama,Jefferson County,6,4.0,0.000050,1,0.0002,0.000052,1,0.0007,"MULTIPOLYGON (((-86.90317 33.47177, -86.90284 ..."
4,01073004000,1,Alabama,Jefferson County,11,5.0,0.000062,1,0.0001,0.000079,1,0.0001,"MULTIPOLYGON (((-86.85463 33.48754, -86.85554 ..."


In [ ]:
# Intermediate data output
# geoda_small.to_file(os.path.join(base_dir, "geoda-dec-2025", "geoda-combined-.geojson"), driver="GeoJSON")

In [7]:
# Use the ID column to fix the sign of gistar and p-values
geoda_test = geoda_small.copy()

geoda_test.loc[geoda_test['id_burd'] == 2, 'p_burd'] = -geoda_test['p_burd']
geoda_test.loc[geoda_test['id_burd'] == 1, 'p_burd'] = geoda_test['p_burd']

geoda_test.loc[geoda_test['id_ind'] == 2, 'p_ind'] = -geoda_test['p_ind']
geoda_test.loc[geoda_test['id_ind'] == 1, 'p_ind'] = geoda_test['p_ind']

geoda_test.loc[geoda_test['id_burd'] == 2, 'g_burd'] = -geoda_test['g_burd']
geoda_test.loc[geoda_test['id_burd'] == 1, 'g_burd'] = geoda_test['g_burd']

geoda_test.loc[geoda_test['id_ind'] == 2, 'g_ind'] = -geoda_test['g_ind']
geoda_test.loc[geoda_test['id_ind'] == 1, 'g_ind'] = geoda_test['g_ind']

In [ ]:
# Create binned classifications for cluster - burdens
burd_case_list = [
  ((geoda_test['p_burd'] > 0.05) | (geoda_test['p_burd'] < -0.05), 'Not Significant'),
  ((geoda_test['p_burd'] <= 0.05) & (geoda_test['p_burd'] > 0.01), 'Hot Spot'),
  ((geoda_test['p_burd'] <= 0.01) & (geoda_test['p_burd'] >= 0), 'Very Hot Spot'),
  ((geoda_test['p_burd'] >= -0.05) & (geoda_test['p_burd'] < -0.01), 'Cold Spot'),
  ((geoda_test['p_burd'] >= -0.01) & (geoda_test['p_burd'] < 0), 'Very Cold Spot'),
]

# Create binned classifications for cluster - indicators
ind_case_list = [
  ((geoda_test['p_ind'] > 0.05) | (geoda_test['p_ind'] < -0.05), 'Not Significant'),
  ((geoda_test['p_ind'] <= 0.05) & (geoda_test['p_ind'] > 0.01), 'Hot Spot'),
  ((geoda_test['p_ind'] <= 0.01) & (geoda_test['p_ind'] >= 0), 'Very Hot Spot'),
  ((geoda_test['p_ind'] >= -0.05) & (geoda_test['p_ind'] < -0.01), 'Cold Spot'),
  ((geoda_test['p_ind'] >= -0.01) & (geoda_test['p_ind'] < 0), 'Very Cold Spot'),
]

In [ ]:
geoda_test['cluster_burd'] = pd.Series(index=geoda_test.index, dtype=object)
geoda_test['cluster_burd'] = geoda_test['cluster_burd'].case_when(caselist=burd_case_list)

geoda_test['cluster_ind'] = pd.Series(index=geoda_test.index, dtype=object)
geoda_test['cluster_ind'] = geoda_test['cluster_ind'].case_when(caselist=ind_case_list)

In [11]:
# Calculate census tract stats
# Total burdens
print(f"Census tracts by total burdens: \n {geoda_test.groupby('CC')['GEOID10'].count()} \n Percentage: \n {(geoda_test.groupby('CC')['GEOID10'].count() / geoda_test.shape[0]) * 100} \n")

# Total indicators
print(f"Census tracts by total indicators: \n {geoda_test.groupby('TC')['GEOID10'].count()} \n Percentage: \n {(geoda_test.groupby('TC')['GEOID10'].count() / geoda_test.shape[0]) * 100} \n")

# Total cluster burdens
print(f"Census tracts by burden cluster: \n {geoda_test.groupby('cluster_burd')['GEOID10'].count()} \n Percentage: \n {(geoda_test.groupby('cluster_burd')['GEOID10'].count() / geoda_test.shape[0]) * 100} \n")

# Total cluster indicators
print(f"Census tracts by ind cluster: \n {geoda_test.groupby('cluster_ind')['GEOID10'].count()} \n Percentage: \n {(geoda_test.groupby('cluster_ind')['GEOID10'].count() / geoda_test.shape[0]) * 100}")

# Original DAC non-DAC classification
# Total cluster indicators
print(f"Census tracts by ind cluster: \n {geoda_test.groupby('SN_C')['GEOID10'].count()} \n Percentage: \n {(geoda_test.groupby('SN_C')['GEOID10'].count() / geoda_test.shape[0]) * 100}")


Census tracts by total burdens: 
 CC
0.0    47333
1.0     5789
2.0     5158
3.0     4982
4.0     4426
5.0     3582
6.0     1893
7.0      539
8.0       65
Name: GEOID10, dtype: int64 
 Percentage: 
 CC
0.0    64.165548
1.0     7.847683
2.0     6.992287
3.0     6.753697
4.0     5.999973
5.0     4.855830
6.0     2.566188
7.0     0.730679
8.0     0.088115
Name: GEOID10, dtype: float64 

Census tracts by total indicators: 
 TC
0     47404
1      4959
2      4140
3      3400
4      2922
5      2456
6      1925
7      1602
8      1427
9      1100
10      924
11      674
12      411
13      250
14       98
15       48
16       16
17        8
18        3
Name: GEOID10, dtype: int64 
 Percentage: 
 TC
0     64.261797
1      6.722518
2      5.612266
3      4.609107
4      3.961121
5      3.329402
6      2.609568
7      2.171703
8      1.934469
9      1.491182
10     1.252593
11     0.913688
12     0.557160
13     0.338905
14     0.132851
15     0.065070
16     0.021690
17     0.010845
18     0.00